In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"
# If charts do not display (for example in VS Code or Colab), uncomment one line:
# pio.renderers.default = "notebook_connected"
# pio.renderers.default = "colab"

print("Plotly version:", __import__("plotly").__version__)

Plotly version: 5.24.1


In [5]:
df = pd.read_csv("GitHub_Repository_Analytics_Dataset_1000_2.csv")
print("Shape:", df.shape)
df.head()

Shape: (1000, 13)


,repository_id,repository_name,owner,stars,forks,watchers,open_issues,contributors,commits,pull_requests,primary_language,created_date,last_updated
0,100001,project_1,dev_190,42556,8430,20668,1551,66,100927,250,C++,2025-10-05,2027-12-09
1,100002,project_2,dev_261,51674,3473,18433,505,247,162993,6311,C++,2023-02-19,2024-01-21
2,100003,project_3,dev_83,24799,13566,21321,780,191,90347,4440,Python,2023-08-15,2024-03-25
3,100004,project_4,dev_264,57998,7388,19918,2035,545,46837,5688,C#,2025-05-15,2027-03-08
4,100005,project_5,dev_20,68215,6203,2931,314,427,133513,9935,TypeScript,2022-10-01,2023-10-09


In [6]:
summary = pd.DataFrame({"dtype": df.dtypes.astype(str),
                        "missing": df.isna().sum(),
                        "missing_%": (df.isna().mean() * 100).round(1)})
summary

,dtype,missing,missing_%
repository_id,int64,0,0.0
repository_name,object,0,0.0
owner,object,0,0.0
stars,int64,0,0.0
forks,int64,0,0.0
watchers,int64,0,0.0
open_issues,int64,0,0.0
contributors,int64,0,0.0
commits,int64,0,0.0
pull_requests,int64,0,0.0


In [8]:
df[["stars", "forks", "watchers", "open_issues", "contributors", "commits", "pull_requests"]].describe().round(2)

,stars,forks,watchers,open_issues,contributors,commits,pull_requests
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,40036.56,7535.80,14732.68,1260.76,294.89,94514.92,5096.76
std,22788.84,4321.93,8721.42,727.23,170.67,58140.86,2913.10
min,49.00,31.00,5.00,0.00,2.00,377.00,22.00
25%,20933.50,3777.25,7066.25,641.25,152.00,45403.50,2639.50
50%,39394.00,7555.00,14579.00,1240.50,296.50,90659.00,4938.50
75%,60195.75,11255.50,22046.75,1900.00,434.50,145206.50,7680.00
max,79974.00,14990.00,29980.00,2498.00,599.00,199790.00,9989.00


In [10]:
df["created_date"] = pd.to_datetime(df["created_date"], format="%Y-%m-%d")
df["last_updated"] = pd.to_datetime(df["last_updated"], format="%Y-%m-%d")

df["created_year"] = df["created_date"].dt.year

# Sort by creation date
df = df.sort_values("created_date").reset_index(drop=True)

print("Repository Creation Date Range:", df["created_date"].min().date(), "to", df["created_date"].max().date())
print("Repository Last Updated Date Range:", df["last_updated"].min().date(), "to", df["last_updated"].max().date())

Repository Creation Date Range: 2018-01-07 to 2026-03-17
Repository Last Updated Date Range: 2018-01-19 to 2028-02-19


In [12]:
# Analyze average stars and open issues grouped by the repository creation year
df.groupby("created_year")[["stars", "open_issues"]].mean().round(0).T

created_year,2018,2019,2020,2021,2022,2023,2024,2025,2026
stars,45405.0,39060.0,42440.0,36206.0,37663.0,41039.0,39690.0,39385.0,42227.0
open_issues,1378.0,1238.0,1302.0,1156.0,1325.0,1183.0,1212.0,1317.0,1221.0


In [14]:
fig1 = px.histogram(df, x="stars",
                    title="Distribution of GitHub Repository Stars",
                    labels={"stars": "Stars"})

# Adjust bins to group by 5,000 star intervals for clean distribution visibility
fig1.update_traces(xbins=dict(start=0, end=80000, size=5000))
fig1.update_layout(yaxis_title="Number of Repositories", bargap=0.05,
                   xaxis=dict(rangeslider=dict(visible=True, thickness=0.12)))
fig1.show()

In [16]:
fig2 = px.scatter(df, x="stars", y="forks", color="primary_language",
                  hover_name="repository_name",
                  hover_data={"owner": True, "open_issues": True, "commits": True},
                  opacity=0.75,
                  labels={"stars": "Stars", "forks": "Forks", "primary_language": "Language"},
                  title="GitHub Stars vs. Forks by Programming Language")

fig2.update_layout(xaxis=dict(rangeslider=dict(visible=True, thickness=0.12)))
fig2.show()

In [18]:
fig3 = px.box(df, x="primary_language", y="stars", color="primary_language",
              points="all", hover_name="repository_name",
              hover_data={"owner": True, "forks": True, "open_issues": True},
              labels={"primary_language": "Programming Language", "stars": "Stars"},
              title="Distribution of Repository Stars by Programming Language")

fig3.update_traces(marker=dict(size=4, opacity=0.6), jitter=0.4)
fig3.update_layout(showlegend=False)
fig3.show()

In [20]:
# Calculate the median stars, forks, and commits grouped by programming language
df.groupby("primary_language")[["stars", "forks", "commits"]].median().round(2)

,stars,forks,commits
primary_language,,,
C#,40767.0,7658.0,82250.0
C++,43534.5,7520.0,85756.0
Go,39826.0,7357.0,94955.0
Java,33571.0,8221.0,99261.0
JavaScript,33532.0,7474.0,88042.0
Python,44203.0,8508.0,92791.0
Rust,41034.0,7108.0,81736.0
TypeScript,36879.0,7046.0,87521.5


In [22]:
fig4 = px.scatter_3d(df, x="stars", y="forks", z="commits", color="primary_language",
                     hover_name="repository_name",
                     hover_data={"owner": True, "open_issues": True},
                     opacity=0.75,
                     labels={"stars": "Stars", "forks": "Forks",
                             "commits": "Commits", "primary_language": "Language"},
                     title="GitHub Stars, Forks, and Commits in 3D (Rotate and Zoom)")
fig4.update_traces(marker=dict(size=3.5))
fig4.update_layout(height=650)
fig4.show()

In [24]:
# Sort the dataset by year to ensure chronological animation order
anim_df = df.sort_values("created_year")
years = sorted(anim_df["created_year"].unique())

# Create an animated scatter plot showing Stars vs. Forks over the years
fig5 = px.scatter(anim_df, x="stars", y="forks", color="primary_language",
                  animation_frame="created_year",
                  hover_name="repository_name",
                  hover_data={"owner": True, "commits": True},
                  range_x=[0, df["stars"].max() * 1.1],
                  range_y=[0, df["forks"].max() * 1.1],
                  labels={"stars": "Stars", "forks": "Forks",
                          "primary_language": "Language", "created_year": "Created Year"},
                  title="GitHub Stars vs. Forks Over the Years (Animation)")

# Adjust animation speed (frame duration in milliseconds)
if fig5.layout.updatemenus:
    fig5.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 1000

fig5.show()

In [26]:
# Count the number of repositories created each year
df.groupby("created_year").size().rename("repositories").to_frame().T

created_year,2018,2019,2020,2021,2022,2023,2024,2025,2026
repositories,109,121,116,128,116,113,142,135,20


In [28]:
# Define GitHub metrics to swap dynamically on the Y-axis
metrics = {
    "forks": "Forks",
    "watchers": "Watchers",
    "commits": "Commits",
    "pull_requests": "Pull Requests"
}

# Initialize with 'forks' as the default Y-axis metric
fig6 = px.scatter(
    df, x="stars", y="forks", color="primary_language",
    hover_name="repository_name",
    hover_data={"owner": True},
    opacity=0.7,
    labels={"stars": "Stars", "forks": "Forks", "primary_language": "Language"},
    title="GitHub Stars vs. Forks"
)

# Create dropdown buttons to dynamically update the Y-axis data and layout labels
buttons = []
for col, label in metrics.items():
    buttons.append(dict(
        label=label,
        method="update",
        args=[
            {"y": [df[col]]},  # Update the Y-axis data
            {
                "yaxis.title.text": label,  # Update the Y-axis label
                "title.text": f"GitHub Stars vs. {label}"  # Update chart title
            }
        ]
    ))

# Add the dropdown menu to the figure layout
fig6.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=1.0,
        xanchor="right",
        y=1.15,
        yanchor="top",
        showactive=True
    )],
    annotations=[dict(
        text="y-axis:",
        x=0.915,
        xref="paper",
        y=1.12,
        yref="paper",
        showarrow=False,
        xanchor="right"
    )]
)

fig6.show()

In [30]:
df[["stars", "forks", "watchers", "commits", "pull_requests"]].corr(method="spearman").loc["stars"].round(2)

,stars
stars,1.00
forks,-0.06
watchers,0.01
commits,0.01
pull_requests,-0.03


In [31]:
figs = [fig1, fig2, fig3, fig4, fig5, fig6]
with open("Lab6_interactive_charts.html", "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>Lab 6: Interactive charts</title></head><body>")
    for i, fig in enumerate(figs):
        f.write(fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False,
                            auto_play=False))
    f.write("</body></html>")
print("Saved Lab6_interactive_charts.html")

Saved Lab6_interactive_charts.html
